# Inspect trained VLA QA model

`train_qa.py`로 학습한 체크포인트를 불러와서, 검증용 QA 샘플 몇 개에 대한 모델 답변을 직접 출력해보는 노트북입니다.

아래 셀에서 `OUTPUT_DIR`, `CACHE_DIR`, `QA_DIR`, `QWEN_NAME`만 본인 환경에 맞게 수정한 뒤 순서대로 실행하면 됩니다.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
from pathlib import Path
import sys

import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "vla").exists() and (REPO_ROOT.parent / "vla").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vla.qwen_vla import SceneQwenVLA
from vla.train_qa import _ensure_tokenizer, _flatten_batch, _generate_predictions, _move_features_to_device, _resolve_dtype
from vla.qa_dataloader import build_qa_dataloader, _resolve_cache_paths, _split_cache_paths

OUTPUT_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/qa_output")
CACHE_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/qa_cache")
QA_DIR = Path("/zfsauton/scratch/mineuih/waymax_rs/qa_dataset")
QWEN_NAME = "Qwen/Qwen3-0.6B"
FILE_INDICES = None
VALIDATION_FRACTION = 0.04
NUM_EXAMPLES = 5
MAX_PROMPT_LENGTH = 128
MAX_ANSWER_LENGTH = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = _resolve_dtype("bf16" if device.type == "cuda" else "fp16") if device.type == "cuda" else torch.float32
print(f'device={device}, dtype={dtype}')

device=cuda, dtype=torch.bfloat16


In [3]:
def find_latest_checkpoint(output_dir: Path) -> Path:
    ckpt_dir = output_dir / "checkpoints"
    checkpoints = sorted(ckpt_dir.glob("step_*.pt"))
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")
    return checkpoints[-1]

latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)
print(f"Loading checkpoint: {latest_checkpoint}")

model = SceneQwenVLA(qwen_name=QWEN_NAME)
_ensure_tokenizer(model)
checkpoint = torch.load(latest_checkpoint, map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(device=device, dtype=dtype)
model.eval()

print("Model loaded successfully.")
print(f"Checkpoint step: {checkpoint.get('step', 'unknown')}")

Loading checkpoint: /zfsauton/scratch/mineuih/waymax_rs/qa_output/checkpoints/step_00049000.pt


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model loaded successfully.
Checkpoint step: 49000


In [5]:
all_cache_paths = _resolve_cache_paths(str(CACHE_DIR), FILE_INDICES)
train_cache_paths, val_cache_paths = _split_cache_paths(all_cache_paths, VALIDATION_FRACTION)
inspection_cache_paths = val_cache_paths if val_cache_paths else train_cache_paths

NUM_EXAMPLES = 100
print(f"num_cache_paths={len(all_cache_paths)}")
print(f"train_cache_paths={len(train_cache_paths)}")
print(f"val_cache_paths={len(val_cache_paths)}")

loader = build_qa_dataloader(
    str(CACHE_DIR),
    file_indices=None,
    qa_dir=str(QA_DIR),
    batch_size=1,
    shuffle_seed=0,
    num_workers=0,
    pin_memory=False,
    cache_paths=inspection_cache_paths,
)

shown = 0
with torch.inference_mode():
    for batch in loader:
        features, prompts, answers, qa_keys = _flatten_batch(batch)
        if not prompts:
            continue

        remaining = NUM_EXAMPLES - shown
        if remaining <= 0:
            break
        if len(prompts) > remaining:
            prompts = prompts[:remaining]
            answers = answers[:remaining]
            qa_keys = qa_keys[:remaining]
            features = {key: value[:remaining] for key, value in features.items()}

        prompt_ids = model.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LENGTH,
            add_special_tokens=True,
        )["input_ids"].to(device)
        features = _move_features_to_device(features, device, dtype)

        amp_enabled = device.type == "cuda"
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
            predictions = _generate_predictions(
                model,
                features,
                prompt_ids,
                device=device,
                max_new_tokens=MAX_ANSWER_LENGTH,
            )

        for prompt, answer, prediction, q_key in zip(prompts, answers, predictions, qa_keys):
            print(f"[key={q_key}]")
            print(f"Q:   {prompt}")
            print(f"GT:  {answer}")
            print(f"Pred:{prediction}")
            print("-" * 88)
            shown += 1

        if shown >= NUM_EXAMPLES:
            break

num_cache_paths=31
train_cache_paths=30
val_cache_paths=1
[key=num_pedestrian_front]
Q:   Is there any pedestrian in front of the ego vehicle? Answer with yes or no.
GT:  yes
Pred:yes
----------------------------------------------------------------------------------------
[key=num_vehicle_front_same_lane]
Q:   Is there any vehicle in front of the ego vehicle in the same lane? Answer with yes or no.
GT:  no
Pred:no
----------------------------------------------------------------------------------------
[key=has_left_lane]
Q:   Is there a lane on the left of the ego vehicle? Answer with yes or no.
GT:  no
Pred:no**
----------------------------------------------------------------------------------------
[key=num_vehicle_behind_same_lane]
Q:   Is there any vehicle behind the ego vehicle in the same lane? Answer with yes or no.
GT:  yes
Pred:no
----------------------------------------------------------------------------------------
[key=num_vehicle_behind_same_lane]
Q:   Is there any vehicl